<a href="https://colab.research.google.com/github/OdysseusPolymetis/pyOdysseus/blob/main/parallel_data_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
import re
import os
import pandas as pd
import numpy as np
from tqdm import tqdm

In [13]:
def clean_text(text):
    """Nettoie le texte des commandes LaTeX."""
    text = re.sub(r'\\renf\{\}', '', text)

    while '\\ital{' in text:
        text = re.sub(r'\\ital\{([^{}]*)\}', r'\1', text)

    text = re.sub(r'\\q\s*', '', text)
    text = re.sub(r'\\fin', '', text)

    text = re.sub(r'\\[a-zA-Z]+', '', text)

    text = text.strip()

    return text

def extract_pairs_from_file(file_path):
    """Extrait les paires de phrases parallèles d'un fichier LaTeX."""
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            file_content = file.read()

        patterns = [
            r'\\jux\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}',     # Pattern pour \jux{grec}{français}
            r'\\juxta\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}'    # Pattern pour \juxta{grec}{français}
        ]

        all_matches = []
        for pattern in patterns:
            matches = re.findall(pattern, file_content)
            if matches:
                print(f"Fichier {os.path.basename(file_path)}: {len(matches)} paires trouvées avec pattern {pattern}")
                all_matches.extend(matches)

        if not all_matches:
            if "\\let\\juxta=\\jux" in file_content:
                print(f"Fichier {os.path.basename(file_path)}: Détection de '\\let\\juxta=\\jux'")

        clean_matches = [(clean_text(greek), clean_text(french)) for greek, french in all_matches]

        return clean_matches

    except Exception as e:
        print(f"Erreur lors du traitement du fichier {file_path}: {e}")
        return []

def process_folder(folder_path, output_csv, output_tsv, min_length=2, max_length=1000):
    """Traite tous les fichiers .tex dans un dossier et ses sous-dossiers."""
    all_pairs = []

    tex_files = []
    for root, _, files in os.walk(folder_path):
        for file in files:
            if file.endswith('.tex') or file.endswith('.txt'):  # Inclure aussi les .txt
                tex_files.append(os.path.join(root, file))

    print(f"Nombre total de fichiers .tex/.txt trouvés: {len(tex_files)}")

    for file_path in tqdm(tex_files, desc="Traitement des fichiers"):
        pairs = extract_pairs_from_file(file_path)
        all_pairs.extend(pairs)

    df = pd.DataFrame(all_pairs, columns=['greek', 'french'])

    initial_count = len(df)
    df = df[df['greek'].str.len().between(min_length, max_length) &
            df['french'].str.len().between(min_length, max_length)]

    df = df.dropna()
    df = df[(df['greek'] != '') & (df['french'] != '')]

    df.drop_duplicates(inplace=True)

    final_count = len(df)

    if initial_count > final_count:
        print(f"{initial_count - final_count} paires éliminées (vides, trop courtes/longues ou doublons).")

    df.to_csv(output_csv, index=False)
    df.to_csv(output_tsv, sep='\t', index=False)

    print(f"\nTotal: {len(df)} paires de phrases extraites")
    print(f"Résultats sauvegardés dans {output_csv} et {output_tsv}")

    return df

def prepare_finetuning_data(df, train_ratio=0.8, dev_ratio=0.1, output_dir='data'):
    """Prépare les données pour le finetuning en les divisant en ensembles train/dev/test."""
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    df = df.sample(frac=1, random_state=42)

    train_size = int(len(df) * train_ratio)
    dev_size = int(len(df) * dev_ratio)

    train_df = df.iloc[:train_size]
    dev_df = df.iloc[train_size:train_size+dev_size]
    test_df = df.iloc[train_size+dev_size:]

    train_df.to_csv(os.path.join(output_dir, 'train.csv'), index=False)
    train_df.to_csv(os.path.join(output_dir, 'train.tsv'), sep='\t', index=False)

    dev_df.to_csv(os.path.join(output_dir, 'dev.csv'), index=False)
    dev_df.to_csv(os.path.join(output_dir, 'dev.tsv'), sep='\t', index=False)

    test_df.to_csv(os.path.join(output_dir, 'test.csv'), index=False)
    test_df.to_csv(os.path.join(output_dir, 'test.tsv'), sep='\t', index=False)

    print(f"\nEnsembles de données préparés dans {output_dir}:")
    print(f"Train: {len(train_df)} paires")
    print(f"Dev: {len(dev_df)} paires")
    print(f"Test: {len(test_df)} paires")

def test_extraction():

    test_content1 = r"\q \jux{\renf{}Ἄειδε, θεά,}{\renf{}Chante, \ital{déesse},}"
    test_content2 = r"\q \juxta{\renf{}\ital{Ἄειδε}, θεά,}{\renf{}Chante, déesse,}"

    test_content3 = r"\q \jux{τὸν \ital{δὲ} ὄχθοι περιρρέουσι}{les rives \ital{donc} coulent-autour}"

    tests = [
        ("Test simple avec jux et \\ital", test_content1),
        ("Test simple avec juxta et \\ital", test_content2),
        ("Test avec \\ital imbriqué", test_content3)
    ]

    for test_name, content in tests:
        print(f"\n{test_name}:")
        print(f"Original: {content}")

        patterns = [
            r'\\jux\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}',
            r'\\juxta\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}'
        ]

        for pattern in patterns:
            matches = re.findall(pattern, content)
            for match in matches:
                greek, french = match
                print(f"\nAvant nettoyage:")
                print(f"  Grec: {greek}")
                print(f"  Français: {french}")

                clean_greek = clean_text(greek)
                clean_french = clean_text(french)

                print(f"\nAprès nettoyage:")
                print(f"  Grec: {clean_greek}")
                print(f"  Français: {clean_french}")

In [14]:
data_folder = '/content/data'
output_tsv = '/content/data/output.tsv'
output_csv = '/content/data/output.csv'
output_dir = '/content/output'

In [15]:
print(f"Traitement du dossier: {data_folder}")
df = process_folder(data_folder, output_csv, output_tsv)

print("\nAperçu des premières lignes:")
print(df.head(5))

df['greek_length'] = df['greek'].str.len()
df['french_length'] = df['french'].str.len()

print("\nStatistiques sur la longueur des phrases:")
print(f"Grec - Min: {df['greek_length'].min()}, Max: {df['greek_length'].max()}, Moyenne: {df['greek_length'].mean():.2f}")
print(f"Français - Min: {df['french_length'].min()}, Max: {df['french_length'].max()}, Moyenne: {df['french_length'].mean():.2f}")

prepare_finetuning_data(df, output_dir=output_dir)

Traitement du dossier: /content/data
Nombre total de fichiers .tex/.txt trouvés: 11


Traitement des fichiers:  36%|███▋      | 4/11 [00:00<00:00, 39.38it/s]

Fichier od05-j.tex: 1253 paires trouvées avec pattern \\juxta\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}
Fichier od04-j.tex: 1446 paires trouvées avec pattern \\jux\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}
Fichier corona-j.tex: 7565 paires trouvées avec pattern \\jux\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}
Fichier od02-j.tex: 1140 paires trouvées avec pattern \\juxta\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}
Fichier odi-j.tex: 1141 paires trouvées avec pattern \\juxta\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}
Fichier od01-j.tex: 1141 paires trouvées avec pattern \\juxta\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}
Fichier il01-j.tex: 1386 paires trouvées avec pattern \\jux\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}
Fichier luc-j 1.tex: 7089 paires trouvées avec pattern \\juxta\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}\{([^{}]*(?:\{

Traitement des fichiers:  82%|████████▏ | 9/11 [00:00<00:00, 44.42it/s]

Fichier lecoq-j.tex: 2226 paires trouvées avec pattern \\juxta\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}


Traitement des fichiers: 100%|██████████| 11/11 [00:00<00:00, 47.47it/s]


Fichier salamis-j.tex: 416 paires trouvées avec pattern \\jux\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}
1897 paires éliminées (vides, trop courtes/longues ou doublons).

Total: 24214 paires de phrases extraites
Résultats sauvegardés dans /content/data/output.csv et /content/data/output.tsv

Aperçu des premières lignes:
                       greek                            french
0              Ἠὼς δὲ ὤρνυτο              Et l'Aurore s'élança
1                  ἐκ λεχέων            hors de <i>son</i> lit
2      παρὰ ἀγαυοῦ Τιθωνοῖο,    d'auprès du magnifique Tithon,
3             ἵνα φέροι φόως  afin qu'elle apportât la lumière
4  ἀθανάτοισι ἠδὲ βροτοῖσιν·    aux immortels et aux mortels ;

Statistiques sur la longueur des phrases:
Grec - Min: 2, Max: 44, Moyenne: 17.36
Français - Min: 2, Max: 63, Moyenne: 23.62

Ensembles de données préparés dans /content/output:
Train: 19371 paires
Dev: 2421 paires
Test: 2422 paires
